# Checking the from-scratch optimizers against `torch.optim`

Same gradients, same initial parameters, float64: the parameters must coincide step by step.

| From scratch (`optimizers.py`) | PyTorch |
|---|---|
| `GradientDescent(lr)` | `SGD(lr)` |
| `Momentum(lr, beta)`: `v = beta*v + (1-beta)*g`, `w -= lr*v` | `SGD(lr * (1 - beta), momentum=beta)`: `buf = beta*buf + g`, `w -= lr*buf` |
| `RMSprop(lr, beta)` | `RMSprop(lr, alpha=beta)` |
| `Adam(lr)` | `Adam(lr)` |

PyTorch's momentum does not average the gradient (no factor 1 - beta), so the same trajectory needs the learning rate multiplied by 1 - beta.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone -b feat/optimization-methods https://github.com/nyawmolny/deep-learning-fundamentals/

Cloning into 'deep-learning-fundamentals'...
remote: Enumerating objects: 327, done.
remote: Counting objects: 100% (327/327), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 327 (delta 76), reused 297 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (327/327), 4.19 MiB | 8.89 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [ ]:
!ls

data.py  model.py  optimizers.py  test_optimizers.py  test_torch.py


In [5]:
import numpy as np
import torch
from torch import nn

from data import make_moons_dataset
from model import initialize_parameters, train
from optimizers import Adam, GradientDescent, Momentum, RMSprop

LR, BETA, DIMS = 0.01, 0.9, [2, 10, 5, 1]

pairs = {
    "gradient descent": (lambda: GradientDescent(LR), lambda p: torch.optim.SGD(p, lr=LR)),
    "momentum": (lambda: Momentum(LR, BETA), lambda p: torch.optim.SGD(p, lr=LR * (1 - BETA), momentum=BETA)),
    "RMSprop": (lambda: RMSprop(LR, BETA), lambda p: torch.optim.RMSprop(p, lr=LR, alpha=BETA, eps=1e-8)),
    "Adam": (lambda: Adam(LR), lambda p: torch.optim.Adam(p, lr=LR, betas=(0.9, 0.999), eps=1e-8)),
}

## 1. Update rules on random gradients (50 steps)

In [6]:
def compare_steps(make_ours, make_torch, steps=50):
    rng = np.random.default_rng(0)
    w0 = rng.normal(size=(3, 2))
    grads = [rng.normal(size=(3, 2)) for _ in range(steps)]

    ours, w = make_ours(), {"W1": w0.copy()}
    for g in grads:
        w = ours.step(w, {"dW1": g})

    t = torch.tensor(w0, dtype=torch.float64, requires_grad=True)
    opt = make_torch([t])
    for g in grads:
        t.grad = torch.tensor(g)
        opt.step()
    return np.abs(w["W1"] - t.detach().numpy()).max()


print(f"{'optimizer':<18} | max |parameter difference|")
for name, (make_ours, make_torch) in pairs.items():
    print(f"{name:<18} | {compare_steps(make_ours, make_torch):.2e}")

optimizer          | max |parameter difference|
gradient descent   | 2.78e-17
momentum           | 5.55e-17
RMSprop            | 0.00e+00
Adam               | 5.55e-17


## 2. End-to-end: the same network trained with both (full batch, 200 epochs)

In [7]:
X, Y = make_moons_dataset(500, noise=0.25, seed=0)
EPOCHS = 200


def torch_costs(make_torch, params0):
    layers, L = [], len(params0) // 2
    for l in range(1, L + 1):
        lin = nn.Linear(params0[f"W{l}"].shape[1], params0[f"W{l}"].shape[0]).double()
        with torch.no_grad():
            lin.weight.copy_(torch.as_tensor(params0[f"W{l}"]))
            lin.bias.copy_(torch.as_tensor(params0[f"b{l}"]).reshape(-1))
        layers.append(lin)
        if l < L:
            layers.append(nn.ReLU())
    model = nn.Sequential(*layers)
    opt, crit = make_torch(model.parameters()), nn.BCEWithLogitsLoss()
    Xt, yt = torch.as_tensor(X.T), torch.as_tensor(Y.T)
    costs = []
    for _ in range(EPOCHS):
        opt.zero_grad()
        loss = crit(model(Xt), yt)
        loss.backward()
        opt.step()
        costs.append(loss.item())
    return np.array(costs)


params0 = initialize_parameters(DIMS, seed=0)
print(f"{'optimizer':<18} | max |cost difference| over {EPOCHS} epochs")
for name, (make_ours, make_torch) in pairs.items():
    _, ours = train(X, Y, DIMS, make_ours(), EPOCHS, batch_size=None, seed=0)
    print(f"{name:<18} | {np.abs(np.array(ours) - torch_costs(make_torch, params0)).max():.2e}")

optimizer          | max |cost difference| over 200 epochs
gradient descent   | 8.88e-16
momentum           | 6.66e-16
RMSprop            | 2.22e-16
Adam               | 8.88e-16


## 3. Notes

All four from-scratch optimizers match `torch.optim` to float64 machine precision: max
parameter difference ~1e-16 after 50 steps, end-to-end cost difference at most 8.9e-16
over 200 full-batch epochs. Momentum needed `lr * (1 - beta)` because PyTorch's
`buf = beta*buf + g` and ours `v = beta*v + (1-beta)*g` differ by that factor. 